In [1]:
# Install latest Ultralytics with YOLOE support
%pip install -q --no-cache-dir "ultralytics>=8.4.0" "ultralytics-thop"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 266.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
CUSTOM_OBJECT_CLASSES = [
    "person",
    "tiger",
    "goat",
    "bird",
    "octopus",
    "squid",
    "crab",
    "cat",
    "fish",
    "shark",
    "cow",
    "horse",
    "turtle",
    "dog",
    "shrimp",
    "beetle",
    "cake",
    "pancake",
    "spring roll",
    "potato chips",
    "strawberry",
    "banana",
    "grape",
    "pineapple",
    "durian",
    "tomato",
    "pomelo",
    "chili pepper",
    "mushroom",
    "asparagus",
    "tofu",
    "meat",
    "vegetable",
    "flower",
    "tree",
    "bicycle",
    "motorcycle",
    "car",
    "truck",
    "bus",
    "boat",
    "canoe",
    "airplane",
    "robot",
    "shirt",
    "pants",
    "dress",
    "raincoat",
    "life jacket",
    "hat",
    "helmet",
    "scarf",
    "glasses",
    "necklace",
    "bracelet",
    "tie",
    "shoes",
    "sandals",
    "slippers",
    "bag",
    "drum",
    "camera",
    "skateboard",
    "roller skates",
    "sword",
    "knife",
    "fire extinguisher",
    "phone",
    "computer",
    "screen",
    "map",
    "compact disc",
    "metal nail",
    "metal sheet",
    "flashlight",
    "candle",
    "fountain",
    "lantern",
    "plate",
    "bowl",
    "cup",
    "spoon",
    "pot",
    "pan",
    "stove",
    "oven",
    "tray",
    "box",
    "sack",
    "basket",
    "table",
    "chair",
    "bench",
    "bookshelf",
    "book",
    "document",
    "tarp",
    "fence",
    "picture frame",
    "canvas",
    "carpet",
    "mat",
    "door",
    "gate",
    "bridge",
    "column",
    "building",
    "skyscraper",
    "house",
    "stage",
    "watch",
    "traffic light",
    "traffic sign"
]


In [3]:
import os
# Prevent CUDA memory fragmentation on PyTorch allocator
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from collections import Counter
from pathlib import Path
import gc
import json
import threading
import time

from IPython.display import FileLink, display
import PIL
import matplotlib
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
import ultralytics
from ultralytics import YOLOE
from ultralytics.utils import ROOT, YAML

print("ultralytics:", ultralytics.__version__)
print("Pillow:", PIL.__version__)
print("matplotlib:", matplotlib.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

# Detect GPUs and configure multi-GPU devices
NUM_GPUS = torch.cuda.device_count()
if NUM_GPUS > 1:
    DEVICES = [f"cuda:{i}" for i in range(NUM_GPUS)]
    print(f"Multi-GPU enabled: {NUM_GPUS} GPUs available -> {DEVICES}")
    for i in range(NUM_GPUS):
        vram_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({vram_gb:.1f} GB VRAM)")
elif NUM_GPUS == 1:
    DEVICES = ["cuda:0"]
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Single-GPU enabled: {DEVICES[0]} ({torch.cuda.get_device_name(0)}, {vram_gb:.1f} GB VRAM)")
else:
    DEVICES = ["cpu"]
    print("Warning: No GPU available, falling back to CPU.")

# Search child video folders too when True.
RECURSIVE = True

# Process every image in the folder. Set an integer for a smaller demo.
MAX_IMAGES = None

# Vocabulary options:
# - "prompt_free": all classes embedded in yoloe-26l-seg-pf.pt
# - "objects365": controlled object-only text prompts
# - "custom" : custom object classes
VOCABULARY_MODE = "custom"

# Inference settings (Strictly sized for 16GB Tesla T4 to avoid OOM)
IMAGE_SIZE = 640
# BATCH_SIZE strictly controls how many images are passed to model.predict per call.
# Batch size 8 uses ~1.8-2.2 GB VRAM per GPU (safe on 14.56 GB T4 with >12 GB headroom).
# If you encounter any constraints, reduce to 4 or 2.
BATCH_SIZE = 8
CONFIDENCE = 0.30
IOU_THRESHOLD = 0.60
MAX_DETECTIONS = 100
USE_HALF = False      # FP32 avoids dtype mismatch with text-prompt embeddings

# Output directories
VISUALIZATION_DIR = Path("/kaggle/working/yoloe_visualizations")
JSON_OUTPUT_DIR = Path("/kaggle/working/yoloe_json")

# Summary settings
SUMMARY_MAX_CLASSES = 12


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
ultralytics: 8.4.162
Pillow: 11.3.0
matplotlib: 3.10.0
torch: 2.10.0+cu128
cuda available: True
Multi-GPU enabled: 2 GPUs available -> ['cuda:0', 'cuda:1']
  GPU 0: Tesla T4 (14.6 GB VRAM)
  GPU 1: Tesla T4 (14.6 GB VRAM)


In [4]:
def canonical_object_name(raw_name):
    return (
        str(raw_name)
        .split("/", maxsplit=1)[0]
        .replace("_", " ")
        .lower()
        .strip()
    )


if VOCABULARY_MODE == "prompt_free":
    MODEL_WEIGHTS = "yoloe-26l-seg-pf.pt"
    _temp_model = YOLOE(MODEL_WEIGHTS)
    raw_model_names = (
        _temp_model.names.items()
        if isinstance(_temp_model.names, dict)
        else enumerate(_temp_model.names)
    )
    DISPLAY_NAMES = {
        int(class_id): canonical_object_name(class_name)
        for class_id, class_name in raw_model_names
    }
    TARGET_CLASSES = None
    del _temp_model
    gc.collect()

    print("Mode: prompt-free")
    print("Checkpoint vocabulary size:", len(DISPLAY_NAMES))
    print("First 30 labels:", list(DISPLAY_NAMES.values())[:30])

elif VOCABULARY_MODE == "objects365":
    MODEL_WEIGHTS = "yoloe-26l-seg.pt"
    objects365_config = YAML.load(ROOT / "cfg/datasets/Objects365.yaml")
    raw_names = objects365_config["names"]

    if isinstance(raw_names, dict):
        raw_names = [
            class_name
            for _, class_name in sorted(
                raw_names.items(),
                key=lambda item: int(item[0]),
            )
        ]

    EXCLUDED_OBJECTS365_CLASSES = {
        "other shoes",
        "other fish",
        "other balls",
        "soccer",
        "tennis",
        "billiards",
        "curling",
        "skating and skiing shoes",
        "french",
    }

    OBJECT_CLASSES = []
    for raw_name in raw_names:
        class_name = canonical_object_name(raw_name)
        if class_name not in EXCLUDED_OBJECTS365_CLASSES:
            OBJECT_CLASSES.append(class_name)

    assert 300 <= len(OBJECT_CLASSES) <= 365
    assert len(OBJECT_CLASSES) == len(set(OBJECT_CLASSES))

    TARGET_CLASSES = OBJECT_CLASSES
    DISPLAY_NAMES = {
        class_id: class_name
        for class_id, class_name in enumerate(OBJECT_CLASSES)
    }

    print("Mode: controlled Objects365 prompts")
    print("Prompt count:", len(OBJECT_CLASSES))
    print("First 30 prompts:", OBJECT_CLASSES[:30])

elif VOCABULARY_MODE == "custom":
    MODEL_WEIGHTS = "yoloe-26l-seg.pt"
    TARGET_CLASSES = CUSTOM_OBJECT_CLASSES
    DISPLAY_NAMES = {
        class_id: class_name
        for class_id, class_name in enumerate(TARGET_CLASSES)
    }

    print("Mode: custom prompts")
    print("Prompt count:", len(TARGET_CLASSES))
    print("Prompts:", TARGET_CLASSES)

else:
    raise ValueError(
        f"Unknown VOCABULARY_MODE={VOCABULARY_MODE!r}. "
        "Use 'prompt_free', 'objects365', or 'custom'."
    )

# Pre-cache checkpoint once in main thread to avoid concurrent download race conditions
print(f"Pre-caching checkpoint '{MODEL_WEIGHTS}'...")
_warmup_model = YOLOE(MODEL_WEIGHTS)
del _warmup_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Model checkpoint ready.")


Mode: custom prompts
Prompt count: 113
Prompts: ['person', 'tiger', 'goat', 'bird', 'octopus', 'squid', 'crab', 'cat', 'fish', 'shark', 'cow', 'horse', 'turtle', 'dog', 'shrimp', 'beetle', 'cake', 'pancake', 'spring roll', 'potato chips', 'strawberry', 'banana', 'grape', 'pineapple', 'durian', 'tomato', 'pomelo', 'chili pepper', 'mushroom', 'asparagus', 'tofu', 'meat', 'vegetable', 'flower', 'tree', 'bicycle', 'motorcycle', 'car', 'truck', 'bus', 'boat', 'canoe', 'airplane', 'robot', 'shirt', 'pants', 'dress', 'raincoat', 'life jacket', 'hat', 'helmet', 'scarf', 'glasses', 'necklace', 'bracelet', 'tie', 'shoes', 'sandals', 'slippers', 'bag', 'drum', 'camera', 'skateboard', 'roller skates', 'sword', 'knife', 'fire extinguisher', 'phone', 'computer', 'screen', 'map', 'compact disc', 'metal nail', 'metal sheet', 'flashlight', 'candle', 'fountain', 'lantern', 'plate', 'bowl', 'cup', 'spoon', 'pot', 'pan', 'stove', 'oven', 'tray', 'box', 'sack', 'basket', 'table', 'chair', 'bench', 'bookshe

In [5]:
# Point this to the root keyframe folder. It should contain many video folders,
# and each video folder should contain keyframe images.
IMAGE_PATH_ROOT = Path("/kaggle/input/datasets/diamondzone/aic2026-batch2-embedding")
INPUT_DIR = Path("/kaggle/input/datasets/diamondzone/aic2026-batch2-embedding/N071-N080/qwen3vl_data/keyframe")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if not INPUT_DIR.exists():
    raise FileNotFoundError(
        f"Input folder does not exist: {INPUT_DIR}\n"
        "Open the Kaggle Input panel and update INPUT_DIR."
    )

iterator = INPUT_DIR.rglob("*") if RECURSIVE else INPUT_DIR.glob("*")
image_paths = sorted(
    path for path in iterator
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

if not image_paths:
    raise FileNotFoundError(f"No supported images found under: {INPUT_DIR}")

if MAX_IMAGES is not None:
    image_paths = image_paths[:MAX_IMAGES]

print(f"Selected {len(image_paths)} image(s)")
for path in image_paths[:10]:
    print(" -", path.relative_to(INPUT_DIR) if path.is_relative_to(INPUT_DIR) else path)

Selected 17193 image(s)
 - N071-V001/keyframe_0.webp
 - N071-V001/keyframe_10032.webp
 - N071-V001/keyframe_10056.webp
 - N071-V001/keyframe_10134.webp
 - N071-V001/keyframe_10170.webp
 - N071-V001/keyframe_10242.webp
 - N071-V001/keyframe_10278.webp
 - N071-V001/keyframe_10338.webp
 - N071-V001/keyframe_10350.webp
 - N071-V001/keyframe_10368.webp


In [6]:
import subprocess
import sys
from pathlib import Path
import time
import torch
import gc

JSON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Create the standalone process-isolated worker script
# Using Path.as_posix() to ensure clean, platform-independent forward slashes
worker_code = r'''
import argparse
from collections import Counter
import gc
import json
import os
from pathlib import Path
import sys
import time
import torch
from tqdm.auto import tqdm
from ultralytics import YOLOE
from ultralytics.utils import ROOT, YAML

CUSTOM_OBJECT_CLASSES = [
    "person", "tiger", "goat", "bird", "octopus", "squid", "crab", "cat", "fish", "shark",
    "cow", "horse", "turtle", "dog", "shrimp", "beetle", "cake", "pancake", "spring roll",
    "potato chips", "strawberry", "banana", "grape", "pineapple", "durian", "tomato", "pomelo",
    "chili pepper", "mushroom", "asparagus", "tofu", "meat", "vegetable", "flower", "tree",
    "bicycle", "motorcycle", "car", "truck", "bus", "boat", "canoe", "airplane", "robot",
    "shirt", "pants", "dress", "raincoat", "life jacket", "hat", "helmet", "scarf", "glasses",
    "necklace", "bracelet", "tie", "shoes", "sandals", "slippers", "bag", "drum", "camera",
    "skateboard", "roller skates", "sword", "knife", "fire extinguisher", "phone", "computer",
    "screen", "map", "compact disc", "metal nail", "metal sheet", "flashlight", "candle",
    "fountain", "lantern", "plate", "bowl", "cup", "spoon", "pot", "pan", "stove", "oven",
    "tray", "box", "sack", "basket", "table", "chair", "bench", "bookshelf", "book", "document",
    "tarp", "fence", "picture frame", "canvas", "carpet", "mat", "door", "gate", "bridge",
    "column", "building", "skyscraper", "house", "stage", "watch", "traffic light", "traffic sign"
]

def canonical_object_name(raw_name):
    return (
        str(raw_name)
        .split("/", maxsplit=1)[0]
        .replace("_", " ")
        .lower()
        .strip()
    )

def setup_vocabulary(vocab_mode):
    if vocab_mode == "prompt_free":
        weights = "yoloe-26l-seg-pf.pt"
        temp_model = YOLOE(weights)
        raw_names = (
            temp_model.names.items()
            if isinstance(temp_model.names, dict)
            else enumerate(temp_model.names)
        )
        display_names = {
            int(cid): canonical_object_name(name)
            for cid, name in raw_names
        }
        del temp_model
        gc.collect()
        return weights, None, display_names
    elif vocab_mode == "objects365":
        weights = "yoloe-26l-seg.pt"
        cfg = YAML.load(ROOT / "cfg/datasets/Objects365.yaml")
        raw_names = cfg["names"]
        if isinstance(raw_names, dict):
            raw_names = [v for _, v in sorted(raw_names.items(), key=lambda x: int(x[0]))]
        excluded = {
            "other shoes", "other fish", "other balls", "soccer", "tennis",
            "billiards", "curling", "skating and skiing shoes", "french",
        }
        obj_classes = [canonical_object_name(n) for n in raw_names if canonical_object_name(n) not in excluded]
        display_names = {cid: name for cid, name in enumerate(obj_classes)}
        return weights, obj_classes, display_names
    elif vocab_mode == "custom":
        weights = "yoloe-26l-seg.pt"
        display_names = {cid: name for cid, name in enumerate(CUSTOM_OBJECT_CLASSES)}
        return weights, CUSTOM_OBJECT_CLASSES, display_names
    else:
        raise ValueError(f"Unknown vocab mode: {vocab_mode}")

def relative_keyframe_path(image_path, input_dir):
    image_path = Path(image_path)
    try:
        return image_path.relative_to(input_dir)
    except ValueError:
        return Path(image_path.name)

def output_image_path(image_path, input_dir, root_dir):
    image_path = Path(image_path)
    try:
        return image_path.relative_to(root_dir).as_posix()
    except ValueError:
        try:
            return image_path.relative_to(input_dir).as_posix()
        except ValueError:
            return image_path.as_posix()

def parse_keyframe_index(image_path):
    stem = Path(image_path).stem
    digits = "".join(c for c in stem if c.isdigit())
    return int(digits) if digits else None

def video_id_from_path(image_path, input_dir):
    image_path = Path(image_path)
    try:
        rel = image_path.relative_to(input_dir)
    except ValueError:
        return None
    return rel.parts[0] if len(rel.parts) > 1 else None

def build_detection_json(result, source_image_path, display_names, input_dir, root_dir, summary_max=12):
    image_path = Path(source_image_path)
    boxes = result.boxes
    detections = []

    if boxes is not None and len(boxes) > 0:
        cls_list = boxes.cls.int().cpu().tolist()
        conf_list = boxes.conf.float().cpu().tolist()
        xyxy_list = boxes.xyxy.float().cpu().tolist() if boxes.xyxy is not None else None
        xywh_list = boxes.xywh.float().cpu().tolist() if boxes.xywh is not None else None
        xyxyn_list = boxes.xyxyn.float().cpu().tolist() if boxes.xyxyn is not None else None
        xywhn_list = boxes.xywhn.float().cpu().tolist() if boxes.xywhn is not None else None

        for i, cid in enumerate(cls_list):
            detections.append({
                "class_id": cid,
                "class_name": display_names.get(cid, str(cid)),
                "confidence": conf_list[i],
                "bbox_xyxy": xyxy_list[i] if xyxy_list else None,
                "bbox_xywh": xywh_list[i] if xywh_list else None,
                "bbox_xyxyn": xyxyn_list[i] if xyxyn_list else None,
                "bbox_xywhn": xywhn_list[i] if xywhn_list else None,
            })
        counts = Counter(display_names.get(cid, str(cid)) for cid in cls_list)
    else:
        counts = Counter()

    sorted_counts = sorted(counts.items(), key=lambda x: (-x[1], x[0]))
    visible_counts = sorted_counts[:summary_max]
    summary = ", ".join(f"{name}: {cnt}" for name, cnt in visible_counts)
    if not summary:
        summary = "No objects above the confidence threshold"
    elif len(sorted_counts) > summary_max:
        summary += f", +{len(sorted_counts) - summary_max} more class(es)"

    height, width = result.orig_shape
    num_detections = int(sum(counts.values()))

    return {
        "image_path": output_image_path(image_path, input_dir, root_dir),
        "video_id": video_id_from_path(image_path, input_dir),
        "keyframe_index": parse_keyframe_index(image_path),
        "filename": image_path.name,
        "image_width": int(width),
        "image_height": int(height),
        "model": "yoloe-26l-seg",
        "summary": summary,
        "object_names": sorted(counts),
        "has_objects": num_detections > 0,
        "class_counts": dict(sorted(counts.items())),
        "num_detections": num_detections,
        "detections": detections,
    }

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--worker-id", type=int, required=True)
    parser.add_argument("--num-workers", type=int, required=True)
    parser.add_argument("--input-dir", type=str, required=True)
    parser.add_argument("--image-root", type=str, required=True)
    parser.add_argument("--output-dir", type=str, required=True)
    parser.add_argument("--vocab-mode", type=str, default="custom")
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--img-size", type=int, default=640)
    parser.add_argument("--conf", type=float, default=0.30)
    parser.add_argument("--iou", type=float, default=0.60)
    parser.add_argument("--max-det", type=int, default=100)
    parser.add_argument("--max-images", type=int, default=None)
    parser.add_argument("--recursive", action="store_true", default=True)
    args = parser.parse_args()

    input_dir = Path(args.input_dir)
    image_root = Path(args.image_root)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Discover all keyframes (deterministic sort)
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    it = input_dir.rglob("*") if args.recursive else input_dir.glob("*")
    all_paths = sorted(p for p in it if p.is_file() and p.suffix.lower() in exts)

    if args.max_images is not None:
        all_paths = all_paths[:args.max_images]

    # Interleaved sharding to balance video scenes across workers
    my_paths = all_paths[args.worker_id::args.num_workers]
    print(f"[Worker {args.worker_id}] Assigned {len(my_paths)} / {len(all_paths)} images")

    if not my_paths:
        print(f"[Worker {args.worker_id}] No images to process.")
        return

    # In process-isolated mode with CUDA_VISIBLE_DEVICES set, the visible device is cuda:0
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    dev_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    print(f"[Worker {args.worker_id}] Running on device: {device} ({dev_name})")

    weights_file, target_classes, display_names = setup_vocabulary(args.vocab_mode)
    model = YOLOE(weights_file)
    if device.startswith("cuda"):
        model.to(device)

    if target_classes is not None:
        model.set_classes(target_classes)
        if device.startswith("cuda"):
            model.to(device)

    pbar = tqdm(
        total=len(my_paths),
        desc=f"Worker {args.worker_id}",
        position=args.worker_id,
        leave=True,
    )

    batch_size = args.batch_size
    processed = 0

    for i in range(0, len(my_paths), batch_size):
        batch = my_paths[i:i + batch_size]
        
        with torch.no_grad():
            results = model.predict(
                source=[str(p) for p in batch],
                imgsz=args.img_size,
                conf=args.conf,
                iou=args.iou,
                max_det=args.max_det,
                device=device,
                retina_masks=False,
                agnostic_nms=False,
                verbose=False,
            )

        for src_path, res in zip(batch, results):
            rel_path = relative_keyframe_path(src_path, input_dir)
            json_file = output_dir / rel_path.with_suffix(".json")
            json_file.parent.mkdir(parents=True, exist_ok=True)

            data = build_detection_json(res, src_path, display_names, input_dir, image_root)
            with open(json_file, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)

            del res
            pbar.update(1)
            processed += 1

        del results

        # Periodically empty cache every 50 batches
        if (i // batch_size) % 50 == 0:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

    pbar.close()
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"[Worker {args.worker_id}] Finished {processed} images.")

if __name__ == "__main__":
    main()
'''

worker_script_path = Path("/kaggle/working/yoloe_worker.py")
worker_script_path.write_text(worker_code, encoding="utf-8")
print(f"Worker script saved to {worker_script_path}")

# 2. Clean memory before spawning subprocesses
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 3. Launch isolated worker processes (1 dedicated process per GPU)
num_workers = NUM_GPUS if NUM_GPUS > 0 else 1
processes = []
start_time = time.time()

print(f"Launching {num_workers} isolated worker process(es)...")
for worker_id in range(num_workers):
    env = os.environ.copy()
    if NUM_GPUS > 0:
        env["CUDA_VISIBLE_DEVICES"] = str(worker_id)
        
    cmd = [
        sys.executable, str(worker_script_path),
        "--worker-id", str(worker_id),
        "--num-workers", str(num_workers),
        "--input-dir", str(INPUT_DIR),
        "--image-root", str(IMAGE_PATH_ROOT),
        "--output-dir", str(JSON_OUTPUT_DIR),
        "--vocab-mode", str(VOCABULARY_MODE),
        "--batch-size", str(BATCH_SIZE),
        "--img-size", str(IMAGE_SIZE),
        "--conf", str(CONFIDENCE),
        "--iou", str(IOU_THRESHOLD),
        "--max-det", str(MAX_DETECTIONS),
    ]
    if MAX_IMAGES is not None:
        cmd.extend(["--max-images", str(MAX_IMAGES)])
    if RECURSIVE:
        cmd.append("--recursive")
        
    p = subprocess.Popen(cmd, env=env)
    processes.append(p)

# Wait for all workers to finish
for p in processes:
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"Worker process failed with exit code {p.returncode}")

elapsed = time.time() - start_time
fps = len(image_paths) / elapsed if elapsed > 0 else 0
print(f"\nAll workers completed in {elapsed:.1f}s ({fps:.2f} img/s)!")

# 4. Verification step: verify NO keyframes were missed
output_json_files = list(JSON_OUTPUT_DIR.rglob("*.json"))
print(f"==========================================")
print(f"Total source images: {len(image_paths)}")
print(f"Total JSON files generated: {len(output_json_files)}")
if len(output_json_files) == len(image_paths):
    print("SUCCESS: 100% of keyframes processed without any missing files!")
else:
    missing = len(image_paths) - len(output_json_files)
    print(f"WARNING: {missing} keyframe(s) were not found in output directory.")
print(f"==========================================")


Worker script saved to /kaggle/working/yoloe_worker.py
Launching 2 isolated worker process(es)...
[Worker 1] Assigned 8596 / 17193 images
[Worker 1] Running on device: cuda:0 (Tesla T4)
requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
[Worker 0] Assigned 8597 / 17193 images
[Worker 0] Running on device: cuda:0 (Tesla T4)
requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 36 packages in 676ms
   Updating https://github.com/ultralytics/CLIP.git (HEAD)
    Updated https://github.com/ultralytics/CLIP.git (a13192f8cb767260d7dfd98c843b0716593169e7)
   Building clip @ git+https://github.com/ultralytics/CLIP.git@a13192f8cb767260d7dfd98c843b0716593169e7
      Built clip @ git+https://github.com/ultralytics/CLIP.git@a13192f8cb767260d7dfd98c843b0716593169e7
Prepared 2 packages in 2.48s
Installed 2 package


Worker 0: 100%|██████████| 8597/8597 [07:43<00:00, 18.55it/s]

Worker 1:  94%|█████████▍| 8097/8596 [07:43<00:27, 18.25it/s]

[Worker 0] Finished 8597 images.



Worker 1: 100%|██████████| 8596/8596 [08:12<00:00, 17.45it/s]


[Worker 1] Finished 8596 images.

All workers completed in 531.7s (32.33 img/s)!
Total source images: 17193
Total JSON files generated: 17193
SUCCESS: 100% of keyframes processed without any missing files!


In [7]:
!zip -r N071-N080.zip /kaggle/working/yoloe_json
from IPython.display import FileLink
FileLink(r'N071-N080.zip')


  adding: kaggle/working/yoloe_json/ (stored 0%)
  adding: kaggle/working/yoloe_json/N077-V003/ (stored 0%)
  adding: kaggle/working/yoloe_json/N077-V003/keyframe_8652.json (deflated 70%)
  adding: kaggle/working/yoloe_json/N077-V003/keyframe_14334.json (deflated 71%)
  adding: kaggle/working/yoloe_json/N077-V003/keyframe_11526.json (deflated 72%)
  adding: kaggle/working/yoloe_json/N077-V003/keyframe_11790.json (deflated 72%)
  adding: kaggle/working/yoloe_json/N077-V003/keyframe_6708.json (deflated 72%)
  adding: kaggle/working/yoloe_json/N077-V003/keyframe_2502.json (deflated 72%)
  adding: kaggle/working/yoloe_json/N077-V003/keyframe_8484.json (deflated 71%)
  adding: kaggle/working/yoloe_json/N077-V003/keyframe_1920.json (deflated 72%)
  adding: kaggle/working/yoloe_json/N077-V003/keyframe_708.json (deflated 70%)
  adding: kaggle/working/yoloe_json/N077-V003/keyframe_14958.json (deflated 74%)
  adding: kaggle/working/yoloe_json/N077-V003/keyframe_9168.json (deflated 73%)
  adding:

/kaggle/working/N071-N080.zip